# Kim Mel-Band RoFormer: tach giọng sach tu toan bo file

Notebook nay chay rieng tren Kaggle GPU, tach **toan bo audio** va chi giu stem giọng trong `clean_speech.wav`. Mac dinh dung checkpoint Kim FT2 Bleedless de han che nhac nen lot vao stem giọng.

> Luu y: day la mo hinh tach vocals/nhac, khong phai mo hinh khu moi chuyen dung. `clean_speech.wav` co nghia la giọng da duoc tach khoi nhac nen; tieng on phong, reverb hoac tieng dong co the van con.

In [ ]:
# Cai dat trong kernel Kaggle hien tai. Chay cell nay mot lan.
import shutil
import subprocess
import sys

if shutil.which("ffmpeg") is None or shutil.which("ffprobe") is None:
    subprocess.check_call(["apt-get", "update", "-qq"])
    subprocess.check_call(["apt-get", "install", "-y", "-qq", "ffmpeg"])

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    # RoFormer dung CUDA cua PyTorch; khong can ONNX Runtime GPU.
    # Bo extra [gpu] de tranh ORT doi CUDA 13 trong khi Kaggle dang dung CUDA 12.8.
    "audio-separator>=0.47.0", "onnxruntime==1.23.2", "soundfile>=0.12.1",
])
print("Da cai xong thu vien.")

## Cau hinh
Dat `INPUT_AUDIO` thanh duong dan file trong `/kaggle/input/...`. `CHUNK_DURATION_SECONDS` chi chia noi bo de giam RAM; file dau ra van la mot file day du.

In [ ]:
from pathlib import Path

# SUA DUONG DAN NAY. Vi du: /kaggle/input/my-audio/podcast.mp3
INPUT_AUDIO = Path("/kaggle/input/your-dataset/your-audio.mp3")

WORK_DIR = Path("/kaggle/working/kim_melband_clean_speech")
OUTPUT_DIR = WORK_DIR / "output"
MODEL_DIR = WORK_DIR / "models"
FINAL_AUDIO = OUTPUT_DIR / "clean_speech.wav"
REPORT_PATH = OUTPUT_DIR / "run_report.json"

KIM_MODELS = {
    "base": "vocals_mel_band_roformer.ckpt",
    "ft": "mel_band_roformer_kim_ft_unwa.ckpt",
    "ft2": "mel_band_roformer_kim_ft2_unwa.ckpt",
    "ft2_bleedless": "mel_band_roformer_kim_ft2_bleedless_unwa.ckpt",
    "ft3": "mel_band_roformer_kim_ft3_unwa.ckpt",
}
MODEL_FILENAME = KIM_MODELS["ft2_bleedless"]
CHUNK_DURATION_SECONDS = 300
# native_fp16 da tao NaN/Inf voi Kim FT2 Bleedless tren Kaggle Torch 2.10/cu128.
# autocast giu cac phep toan nhay cam o float32 va van tiet kiem VRAM.
PRECISION_MODE = "autocast"  # "autocast" hoac "fp32"
RETRY_FP32_ON_NONFINITE = True
USE_TORCH_COMPILE = False

AUDIO_EXTENSIONS = {".wav", ".flac", ".mp3", ".m4a", ".aac", ".ogg", ".opus", ".wma"}
assert INPUT_AUDIO.is_file(), f"Khong tim thay INPUT_AUDIO: {INPUT_AUDIO}"
assert INPUT_AUDIO.suffix.lower() in AUDIO_EXTENSIONS, f"Dinh dang audio chua duoc khai bao: {INPUT_AUDIO.suffix}"
print(f"Input: {INPUT_AUDIO}")
print(f"Model: {MODEL_FILENAME}")
print(f"Output duy nhat: {FINAL_AUDIO}")

In [ ]:
# Kiem tra GPU truoc khi tai checkpoint lon.
import importlib.metadata
import torch

print("audio-separator =", importlib.metadata.version("audio-separator"))
print("torch =", torch.__version__)
print("CUDA available =", torch.cuda.is_available())
assert torch.cuda.is_available(), "Hay bat GPU trong Kaggle: Settings > Accelerator > GPU."
print("GPU =", torch.cuda.get_device_name(0))
print("VRAM (GiB) =", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))

In [ ]:
# Doc thong tin input bang ffprobe ma khong nap toan bo file vao RAM.
import json
import subprocess

def probe_audio(path):
    command = [
        "ffprobe", "-v", "error", "-select_streams", "a:0",
        "-show_entries", "stream=sample_rate,channels,duration:format=duration",
        "-of", "json", str(path),
    ]
    payload = json.loads(subprocess.check_output(command, text=True))
    streams = payload.get("streams", [])
    if not streams:
        raise RuntimeError(f"File khong co audio stream: {path}")
    stream = streams[0]
    duration = stream.get("duration") or payload.get("format", {}).get("duration")
    return {
        "duration_seconds": float(duration),
        "sample_rate": int(stream["sample_rate"]),
        "channels": int(stream["channels"]),
    }

input_info = probe_audio(INPUT_AUDIO)
assert input_info["duration_seconds"] > 0, "Input co do dai bang 0."
print(json.dumps(input_info, indent=2))

## Tach giọng
Cell nay se tai checkpoint lan dau, tach toan bo file, chon dung stem co ten `Vocals`, doi ten thanh `clean_speech.wav`, roi xoa stem nhac.

In [ ]:
import gc
import inspect
import os
import shutil
import time
from audio_separator.separator import Separator

# Lam sach rieng thu muc ket qua cua lan chay nay, khong cham vao input.
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

def clear_generated_audio():
    for path in OUTPUT_DIR.rglob("*"):
        if path.is_file() and path.suffix.lower() in AUDIO_EXTENSIONS:
            path.unlink()

def make_separator(precision_mode):
    if precision_mode not in {"autocast", "fp32"}:
        raise ValueError(f"PRECISION_MODE khong hop le: {precision_mode}")
    wanted_kwargs = {
        "output_dir": str(OUTPUT_DIR),
        "model_file_dir": str(MODEL_DIR),
        "output_format": "WAV",
        "use_autocast": precision_mode == "autocast",
        # Khong dung native FP16: log thuc te cho thay checkpoint nay tao NaN/Inf.
        "use_native_fp16": False,
        "use_torch_compile": bool(USE_TORCH_COMPILE and torch.cuda.is_available()),
        "chunk_duration": float(CHUNK_DURATION_SECONDS),
        "normalization_threshold": 1.0,
        # soundfile 0.47 lay nham subtype MPEG_LAYER_III cua input MP3 de ghi WAV.
        # pydub/ffmpeg ghi dung container WAV va khong gap unsupported encoding.
        "use_soundfile": False,
        "log_level": 20,
    }
    accepted = inspect.signature(Separator.__init__).parameters
    separator_kwargs = {key: value for key, value in wanted_kwargs.items() if key in accepted}
    dropped = sorted(set(wanted_kwargs) - set(separator_kwargs))
    if dropped:
        print("Thong bao: ban audio-separator nay khong nhan:", dropped)
    instance = Separator(**separator_kwargs)
    instance.load_model(model_filename=MODEL_FILENAME)
    return instance

torch.cuda.reset_peak_memory_stats()
started = time.perf_counter()
precision_used = PRECISION_MODE
precision_attempts = []
separator = None

while True:
    try:
        clear_generated_audio()
        separator = make_separator(precision_used)
        print("Effective precision =", getattr(separator, "effective_precision", precision_used))
        raw_outputs = separator.separate(str(INPUT_AUDIO))
        precision_attempts.append({"mode": precision_used, "status": "ok"})
        break
    except Exception as exc:
        message = f"{type(exc).__name__}: {exc}"
        precision_attempts.append({"mode": precision_used, "status": "failed", "error": message})
        is_nonfinite = "finite" in str(exc).lower() or "nan" in str(exc).lower()
        if not (RETRY_FP32_ON_NONFINITE and precision_used != "fp32" and is_nonfinite):
            raise
        print(f"Autocast tao du lieu khong huu han ({message}). Dang retry toan bo file bang FP32...")
        del separator
        separator = None
        clear_generated_audio()
        gc.collect()
        torch.cuda.empty_cache()
        precision_used = "fp32"

elapsed_seconds = time.perf_counter() - started
peak_vram_gib = torch.cuda.max_memory_allocated() / 2**30

produced = []
for item in raw_outputs:
    path = Path(item)
    if not path.is_absolute():
        path = OUTPUT_DIR / path
    produced.append(path)

print("Cac stem mo hinh tra ve:")
for path in produced:
    print(" -", path.name)

vocal_candidates = [path for path in produced if path.is_file() and "vocal" in path.name.lower()]
if not vocal_candidates:
    raise RuntimeError(
        "Mo hinh khong tra ve stem Vocals. Khong tu dong lay output dau tien vi co the do la nhac. "
        f"Outputs: {[path.name for path in produced]}"
    )

# Neu thu vien tra nhieu stem vocal, uu tien file lon nhat thay vi dua vao thu tu khong duoc bao dam.
vocal_path = max(vocal_candidates, key=lambda path: path.stat().st_size)
if FINAL_AUDIO.exists():
    FINAL_AUDIO.unlink()
shutil.move(str(vocal_path), str(FINAL_AUDIO))

# Yeu cau cua notebook: chi de lai mot audio speech, xoa tat ca stem/audio phu.
for path in OUTPUT_DIR.rglob("*"):
    if path.is_file() and path.suffix.lower() in AUDIO_EXTENSIONS and path.resolve() != FINAL_AUDIO.resolve():
        path.unlink()

assert FINAL_AUDIO.is_file() and FINAL_AUDIO.stat().st_size > 0
print(f"Hoan tat sau {elapsed_seconds / 60:.2f} phut")
print(f"Peak VRAM do PyTorch ghi nhan: {peak_vram_gib:.2f} GiB")
print(f"Audio sach: {FINAL_AUDIO}")

## Kiem tra ket qua
Kiem tra thoi luong, NaN/Inf, peak va RMS theo block de khong ton RAM voi file dai.

In [ ]:
import math
import numpy as np
import soundfile as sf

def scan_wav(path, block_frames=1_048_576):
    total_samples = 0
    sum_squares = 0.0
    peak = 0.0
    all_finite = True
    with sf.SoundFile(path) as audio_file:
        metadata = {
            "sample_rate": audio_file.samplerate,
            "channels": audio_file.channels,
            "frames": len(audio_file),
            "duration_seconds": len(audio_file) / audio_file.samplerate,
            "subtype": audio_file.subtype,
        }
        while True:
            block = audio_file.read(block_frames, dtype="float32", always_2d=True)
            if block.size == 0:
                break
            all_finite = all_finite and bool(np.isfinite(block).all())
            peak = max(peak, float(np.max(np.abs(block))))
            sum_squares += float(np.sum(block.astype(np.float64) ** 2))
            total_samples += block.size
    metadata["peak"] = peak
    metadata["rms"] = math.sqrt(sum_squares / max(total_samples, 1))
    metadata["all_finite"] = all_finite
    return metadata

output_info = scan_wav(FINAL_AUDIO)
duration_error = abs(output_info["duration_seconds"] - input_info["duration_seconds"])
assert output_info["frames"] > 0, "Output rong."
assert output_info["all_finite"], "Output co NaN hoac Inf."
assert output_info["peak"] > 1e-5, "Output gan nhu im lang hoan toan."
assert duration_error <= max(1.0, input_info["duration_seconds"] * 0.001), (
    f"Output lech thoi luong qua lon: {duration_error:.3f}s"
)

report = {
    "input": str(INPUT_AUDIO),
    "output": str(FINAL_AUDIO),
    "model": MODEL_FILENAME,
    "chunk_duration_seconds": CHUNK_DURATION_SECONDS,
    "precision_requested": PRECISION_MODE,
    "precision_used": precision_used,
    "precision_attempts": precision_attempts,
    "elapsed_seconds": round(elapsed_seconds, 3),
    "peak_vram_gib": round(peak_vram_gib, 3),
    "input_info": input_info,
    "output_info": output_info,
    "duration_error_seconds": round(duration_error, 6),
}
REPORT_PATH.write_text(json.dumps(report, indent=2), encoding="utf-8")
print(json.dumps(report, indent=2))
print("\nPASS: clean_speech.wav hop le va la audio output duy nhat.")

In [ ]:
# Nghe thu toi da 60 giay dau ma khong tao them file audio.
from IPython.display import Audio, FileLink, display

with sf.SoundFile(FINAL_AUDIO) as audio_file:
    preview_frames = min(len(audio_file), audio_file.samplerate * 60)
    preview = audio_file.read(preview_frames, dtype="float32", always_2d=True)
    preview_rate = audio_file.samplerate
display(Audio(preview.T, rate=preview_rate))
display(FileLink(str(FINAL_AUDIO)))
display(FileLink(str(REPORT_PATH)))

## Neu bi thieu RAM
Giam `CHUNK_DURATION_SECONDS` tu `300` xuong `180` hoac `120`, sau do chay lai tu cell **Tach giọng**. Khong thay doi `segment_size`/`overlap` cua checkpoint neu chua benchmark vi cac gia tri do da nam trong cau hinh model.